# Data preprocessing

First we load the csv we got from our data collection team :))

In [157]:
import pandas as pd
import os

repo_root = os.getcwd()
file_path = os.path.join(repo_root, '..', "Data Collection", "MASTER_ALL_DATA.csv")

raw_df = pd.read_csv(file_path)

# # Add a parameter that tells read_csv to not parse strings as NaN
# df_different_parsing = pd.read_csv(file_path, keep_default_na=False)

raw_df.head(3)


transition = '\n𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 \n'

/var/folders/sz/8v4khwpx0b76jkngnc80xll40000gn/T/ipykernel_56095/3150949563.py:7: DtypeWarning: Columns (0: title, 1: created_utc, 2: permalink) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(file_path)


# 2. Tidy up data format
Cause of NaN Problem: we have vertical stack of data, we need to merge the 3 data sources to have a horizontal (full) dataset to work with

--> re-separate 3 sources, then merge using app_id as the merge id

In [158]:
raw_df = pd.read_csv(file_path, dtype={'app_id': int, 'title': str, 'created_utc': str, 'permalink': str})

#Steam Monthly Player Data
steam_df = raw_df[raw_df['month'].notna()].copy()
steam_df = steam_df[['app_id', 'game_name', 'genre', 'month', 'peak_players', 'avg_players']]

# Reddit engagement data 
reddit_engagement = raw_df[raw_df['engagement'].notna()].copy()
reddit_engagement = reddit_engagement[['app_id', 'engagement', 'valence', 'n_posts']]

# Ensure one row per game app_id
reddit_engagement = reddit_engagement.drop_duplicates(subset=['app_id'])

# (Inner Join) merge using app_id, we only keep data where there is a merge partner
df_analysis = pd.merge(steam_df, reddit_engagement, on='app_id', how='inner')


print(df_analysis.head(3))
df_analysis.info()

    app_id            game_name  \
0  1063730  New World: Aeternum   
1  1063730  New World: Aeternum   
2  1063730  New World: Aeternum   

                                           genre         month peak_players  \
0  Action, Adventure, Massively Multiplayer, RPG  Last 30 Days          987   
1  Action, Adventure, Massively Multiplayer, RPG    April 2026          975   
2  Action, Adventure, Massively Multiplayer, RPG    March 2026        1,093   

  avg_players  engagement  valence  n_posts  
0       550.0         0.0      NaN      0.0  
1       579.3         0.0      NaN      0.0  
2       581.7         0.0      NaN      0.0  
<class 'pandas.DataFrame'>
RangeIndex: 443820 entries, 0 to 443819
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   app_id        443820 non-null  int64  
 1   game_name     443820 non-null  str    
 2   genre         443820 non-null  str    
 3   month         443820 non-null  str

### Setting data types
e.g.


 \#   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   app_id        443820 non-null  int64            <- str
 1   game_name     443820 non-null  str              <- category                  
 2   genre         443820 non-null  str              <- category    
 3   month         443820 non-null  str              <- ?    
 4   peak_players  443820 non-null  str              <- float    
 5   avg_players   443820 non-null  str              <- float             


In [159]:
#our key shouldn't be used for mathematical operations:
df_analysis['app_id'] = df_analysis['app_id'].astype(str)

# Convert string columns to categorical data
categorical_features = ['game_name', 'genre']
df_analysis[categorical_features] = df_analysis[categorical_features].astype("category")

# Let's see how many unique values we have for each of these features
print(df_analysis[categorical_features].nunique())
df_analysis.head()

df_analysis['peak_players'] = df_analysis['peak_players'].astype(str).str.replace(',', '').astype(float)
df_analysis['avg_players'] = df_analysis['avg_players'].astype(str).str.replace(',', '').astype(float)

df_analysis.info()


game_name    1917
genre         359
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 443820 entries, 0 to 443819
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   app_id        443820 non-null  str     
 1   game_name     443820 non-null  category
 2   genre         443820 non-null  category
 3   month         443820 non-null  str     
 4   peak_players  443820 non-null  float64 
 5   avg_players   443820 non-null  float64 
 6   engagement    443820 non-null  float64 
 7   valence       207441 non-null  float64 
 8   n_posts       443820 non-null  float64 
dtypes: category(2), float64(5), str(2)
memory usage: 25.4 MB


#### Investigating month's datatype

Format: 
1             April 2026



- Probably not worth the pain converting to datetime as we'll use the peak of the peaks (doesn't matter when it happens)
- Drop 'Last 30 Days' from df_analysis['month'] to keep overlaps with months from happening.


In [160]:
print(df_analysis['month'].head(5)) #went to check if we should transform to datetime, i say no
df_analysis = df_analysis[df_analysis['month'] != 'Last 30 Days']

print(df_analysis['month'].head(5))

0     Last 30 Days
1       April 2026
2       March 2026
3    February 2026
4     January 2026
Name: month, dtype: str
1       April 2026
2       March 2026
3    February 2026
4     January 2026
5    December 2025
Name: month, dtype: str


So now we have....

In [161]:
nb_games = df_analysis['game_name'].nunique()
genres = df_analysis['genre'].nunique()
print(f"""For analysis the cleaned dataset contains:
      
        -{nb_games} games,
        -{genres} genres,
        -{df_analysis.shape[0]} valid rows
        
        """)

#rows per game /months with peak_player data available
print(f"The top 5 games with the most data on monthly players are:\n\n{df_analysis['game_name'].value_counts().head(5)}")

For analysis the cleaned dataset contains:

        -1917 games,
        -359 genres,
        -439253 valid rows

        
The top 5 games with the most data on monthly players are:

game_name
Heroes & Generals             1224
TrackMania Nations Forever    1162
theHunter Classic             1144
Transformice                  1088
Clicker Heroes                1056
Name: count, dtype: int64


# 3. Checks & fixes before aggreation
See what data we're dealing with at this point & how to aggregate it for our hypotheses.
- check for missing values (so we can decide how to handle them)
- decide how to aggregate our data
- check distribution of values to see which analyses are viable

In [162]:
df = df_analysis.copy() #easier to type and if we mess up we dont affect our clean initial one
df.info()
# What would that dropping NaNs do to our data?
df.dropna().info()
print(transition)

print(df.describe()) #only shows the numerical variables
print(transition)


<class 'pandas.DataFrame'>
Index: 439253 entries, 1 to 443819
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   app_id        439253 non-null  str     
 1   game_name     439253 non-null  category
 2   genre         439253 non-null  category
 3   month         439253 non-null  str     
 4   peak_players  439253 non-null  float64 
 5   avg_players   439253 non-null  float64 
 6   engagement    439253 non-null  float64 
 7   valence       205386 non-null  float64 
 8   n_posts       439253 non-null  float64 
dtypes: category(2), float64(5), str(2)
memory usage: 28.5 MB
<class 'pandas.DataFrame'>
Index: 205386 entries, 121 to 443528
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype   
---  ------        --------------   -----   
 0   app_id        205386 non-null  str     
 1   game_name     205386 non-null  category
 2   genre         205386 non-null  category
 3   month         205386 no

### Dropping all rows containing NaN shows us:
- 207441 rows are kept from the previous 443820

We see that valence is the culprit. ( 7   valence       207441 non-null  float64)
- `valence` Reflects how positively the game is received (1.0 = unanimous upvote, 0.5 = controversial).

When we don't have reddit data, this value is kept empty. We cannot use any values 0-1, because that'd give it a rating on our scale when we do not have information about the valence.

Therefore we will not drop the NaNs, as these contain relevant information! Games can have players even when no engagement happens on r/gaming and we want to include this data for correlation between engagement and player count!

Doublechecking this "0 posts hypothesis" leading to valence of NaN below: When valence is missing, what is the post count distribution?

In [169]:
valence_summary = df_analysis.groupby(df_analysis['valence'].isna())['n_posts'].agg('count')
print(valence_summary, '\n')

print(f"Total rows in original dataframe: {len(df_analysis)}")
print(f"Total rows in valence summary: {valence_summary.sum()}")

valence
False    120064
True     142927
Name: n_posts, dtype: int64 

Total rows in original dataframe: 262991
Total rows in valence summary: 262991


In [164]:
df_analysis['valence'].dropna().describe()

count    205386.000000
mean          0.757790
std           0.150853
min           0.230000
25%           0.681429
50%           0.787778
75%           0.860000
max           1.000000
Name: valence, dtype: float64

The distribution of Valence also looks reasonable, most posts would have more upvotes than downvotes! 

## Isolating 2021-2025 Data
To ensure our Reddit data aligns with our Playerdata, we discard rows outside the defined scope of our analysis (years 2021-2025).

Reddit Data date window: posts whose created_utc falls between 2021-01-01 and 2025-12-31 are kept; everything else is discarded.


In [165]:
unique_months = df_analysis['month'].unique()
print(f"Total unique months in player count data: {len(unique_months)}")

first = unique_months[-1]
last = unique_months[1]

print(f'We have player data from {first} to {last}')

Total unique months in player count data: 166
We have player data from July 2012 to March 2026


In [166]:
df_analysis['year'] = df_analysis['month'].str.split(' ').str[-1].astype(int)
#exclude rows with year being outside 2021-2025 range
df_analysis = df_analysis[(df_analysis['year'] >= 2021) & (df_analysis['year'] <= 2025)]


print(f"Df 2021-2025 contains {df_analysis.shape[0]} rows.")

Df 2021-2025 contains 262991 rows.


## Final Distributions Check

- `player peak`:  huge deviations  (probably accurate)
- `avg_players`: huge deviations  (probably accurate)
- `engagement `:  more than half the games have an engagement score of 0.0 (probably accurate)
- `valence `:  reasonable
- `n_posts `:  most don't have a single post (probably accurate)
- `year`: good

In [184]:
df_analysis.describe()

,peak_players,avg_players,engagement,valence,n_posts,year
count,2.629910e+05,262991.000000,2.629910e+05,120064.000000,262991.000000,262991.000000
mean,5.235105e+03,2536.727599,9.654122e+03,0.748662,3.887654,2023.028183
std,2.756832e+04,12221.942989,5.685308e+04,0.149655,9.101649,1.395902
min,1.000000e+00,0.200000,0.000000e+00,0.230000,0.000000,2021.000000
25%,1.870000e+02,73.050000,0.000000e+00,0.670000,0.000000,2022.000000
50%,6.840000e+02,299.700000,0.000000e+00,0.775385,0.000000,2023.000000
75%,2.276000e+03,1120.600000,6.200000e+01,0.852500,4.000000,2024.000000
max,1.329605e+06,699094.100000,1.622955e+06,1.000000,158.000000,2025.000000


# Aggregating our data

-> one line = one game

We keep the first value from genre, engagement, valence and n_posts as will stay consistent (from our previous merge).

- `genre`: first value 
- `player peak`:  maximum value 
- `avg_players`: mean across all months averages
- `engagement `:  first value
- `valence `:  first value 
- `n_posts `:  first value

In [185]:

grouped_df = df_analysis.groupby(['app_id', 'game_name']).agg({
    'genre': 'first',             
    'peak_players': 'max',        # maximum peak from all peaks (h1)
    'avg_players': 'mean',        # average across all months (yes, the average of the average player counts :p)
    'engagement': 'first',        # 
    'valence': 'first',           
    'n_posts': 'first'            
}).reset_index()

print(f"Grouped dataset contains {grouped_df.shape[0]} unique games.")

grouped_df.describe()

Grouped dataset contains 1921 unique games.


,peak_players,avg_players,engagement,valence,n_posts
count,1.921000e+03,1921.000000,1.921000e+03,785.000000,1921.000000
mean,1.347407e+04,1749.722572,9.877262e+03,0.746100,3.384695
std,6.429960e+04,8653.246582,5.967529e+04,0.148718,8.878424
min,1.000000e+00,0.900000,0.000000e+00,0.230000,0.000000
25%,7.470000e+02,49.988000,0.000000e+00,0.670000,0.000000
50%,1.970000e+03,175.983333,0.000000e+00,0.767143,0.000000
75%,6.340000e+03,662.893333,2.600000e+01,0.852500,3.000000
max,1.329605e+06,232871.853333,1.622955e+06,1.000000,158.000000


In [178]:
#sorted by peak players
print(grouped_df[['game_name', 'peak_players', 'engagement', 'n_posts', 'valence']].sort_values('peak_players', ascending= False).head(3))
print(transition)

#sorted by engagement
print(grouped_df[['game_name', 'peak_players', 'engagement','n_posts', 'valence']].sort_values('engagement', ascending= False).head(3))
print(transition)

#sorted by valence
print(grouped_df[['game_name', 'peak_players', 'engagement', 'n_posts', 'valence']].sort_values('valence', ascending= False).head(5))

                game_name  peak_players  engagement  n_posts   valence
1434  PUBG: BATTLEGROUNDS     1329605.0     81156.0     12.0  0.833333
616              Lost Ark     1324761.0       131.0     20.0  0.748500
69    New World: Aeternum      913027.0         0.0      0.0       NaN

𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 

                     game_name  peak_players  engagement  n_posts   valence
1770  Control Ultimate Edition        4800.0   1622955.0     36.0  0.835000
1096                      DOOM        6416.0    715244.0     21.0  0.910952
112             Cyberpunk 2077      273990.0    674966.0     41.0  0.889024

𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 𓆝 𓆟 𓆞 𓆝 𓆟 

             game_name  peak_players  engagement  n_posts  valence
581            OpenTTD        3196.0         3.0      2.0      1.0
1514        Stoneshard        8595.0         1.0      1.0      1.0
1634     Overcooked! 2       17176.0         3.0      1.0      1.0
1427  SteamWorld Dig 2        3728.0         4.0      2.0 

### Saving our clean aggregated & unaggreated datasets to .csv for further analysis

In [189]:
import os

basepath = os.getcwd()
dirpath = os.path.join(basepath, 'data')

os.mkdir(dirpath)

grouped_df.to_csv('data/games_clean_summary.csv', index=False)
df_analysis.to_csv('data/games_clean_monthly_timeline_2026.csv', index=False)